# Notebook 01: The .explain() API

## Learning Objectives

By completing this notebook, you will:
1. Generate synthetic blog traffic data with publishing and holiday events
2. Train NHITS with exogenous features using NeuralForecast
3. Call `.explain()` with Integrated Gradients
4. Parse the `insample` attribution tensor and identify which lags matter most
5. Visualize lag attributions as a heatmap across the 28-day horizon

## Prerequisites

- Module 01: NHITS and the NeuralForecast API
- Guide `01_explainability_methods.md` (read this first)

## Estimated Time

12–15 minutes

## Setup

```bash
pip install neuralforecast captum
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings("ignore")

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MSE, MAE

np.random.seed(42)
print("Setup complete.")

## 1. Generate Synthetic Blog Traffic Data

We build a daily time series that mimics the dataset from the source material:
- 1000 days of daily visitor counts (2020-01-01 to 2022-09-26)
- Two binary exogenous features: `published` (article published today) and `is_holiday`
- Publishing adds a spike of ~500–800 visitors for 2 days
- Holidays reduce traffic by ~15%
- Weekly seasonality: Monday traffic ~20% lower than Friday
- Additive Gaussian noise

This matches the blog traffic domain described in the source material while using fully synthetic data that can be regenerated deterministically.

In [ ]:
def generate_blog_traffic(n_days=1000, seed=42):
    """
    Generate synthetic daily blog traffic with publishing and holiday events.

    Returns
    -------
    pd.DataFrame
        Columns: ds, unique_id, y, published, is_holiday
        All columns required by NeuralForecast.
    """
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start="2020-01-01", periods=n_days, freq="D")

    # Weekly seasonality: Mon=0.80, Tue=0.90, Wed=1.00, Thu=1.05, Fri=1.10,
    #                     Sat=0.95, Sun=0.85
    day_of_week_multiplier = np.array([0.80, 0.90, 1.00, 1.05, 1.10, 0.95, 0.85])
    seasonal_component = day_of_week_multiplier[dates.dayofweek]

    # Slow upward trend (blog is growing)
    trend = 400 + np.linspace(0, 200, n_days)

    # Publishing events: approximately 3 per week on random weekdays
    published = np.zeros(n_days, dtype=np.float32)
    for i in range(0, n_days, 7):
        # Pick 2-3 weekdays in each week to publish
        n_articles = rng.integers(2, 4)
        weekdays_in_window = [j for j in range(i, min(i + 5, n_days))
                              if dates[j].dayofweek < 5]
        if len(weekdays_in_window) >= n_articles:
            publish_days = rng.choice(weekdays_in_window, size=n_articles, replace=False)
            published[publish_days] = 1.0

    # Holiday mask: US public holidays (simplified)
    holidays = {
        "2020-01-01", "2020-07-04", "2020-11-26", "2020-12-25",
        "2021-01-01", "2021-07-04", "2021-11-25", "2021-12-25",
        "2022-01-01", "2022-07-04"
    }
    is_holiday = np.array(
        [1.0 if str(d.date()) in holidays else 0.0 for d in dates],
        dtype=np.float32
    )

    # Publishing spike: +600 visitors on day of publication, +200 next day
    publishing_lift = np.zeros(n_days)
    publishing_lift += published * 600.0
    # Day-after spillover
    publishing_lift[1:] += published[:-1] * 200.0

    # Holiday effect: -15% of base traffic
    holiday_reduction = is_holiday * (-0.15 * trend)

    # Noise
    noise = rng.normal(0, 40, n_days)

    visitors = trend * seasonal_component + publishing_lift + holiday_reduction + noise
    visitors = np.clip(visitors, 50, None)  # floor at 50 visitors

    df = pd.DataFrame({
        "unique_id": "blog",
        "ds": dates,
        "y": visitors.astype(np.float32),
        "published": published,
        "is_holiday": is_holiday
    })
    return df


df = generate_blog_traffic(n_days=1000)

print(f"Dataset: {len(df)} days ({df['ds'].min().date()} to {df['ds'].max().date()})")
print(f"Publishing events: {int(df['published'].sum())} days")
print(f"Holidays: {int(df['is_holiday'].sum())} days")
print(f"Visitor range: {df['y'].min():.0f} to {df['y'].max():.0f}")
print()
print(df.head(10).to_string(index=False))

## 2. Visualize the Training Data

Before training, always plot the raw series. Verify that publishing spikes are visible as abrupt upward jumps.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Daily visitors
axes[0].plot(df["ds"], df["y"], linewidth=0.8, color="steelblue", label="visitors")
axes[0].set_ylabel("Daily Visitors", fontsize=11)
axes[0].set_title("Synthetic Blog Traffic Dataset", fontsize=13)
axes[0].grid(True, alpha=0.3)

# Publishing events
pub_dates = df[df["published"] == 1]["ds"]
pub_y = df[df["published"] == 1]["y"]
axes[1].vlines(pub_dates, 0, 1, colors="darkorange", alpha=0.5, linewidth=0.8)
axes[1].set_ylabel("Article Published", fontsize=11)
axes[1].set_yticks([0, 1])
axes[1].grid(True, alpha=0.3)

# Holidays
hol_dates = df[df["is_holiday"] == 1]["ds"]
axes[2].vlines(hol_dates, 0, 1, colors="crimson", alpha=0.8, linewidth=2)
axes[2].set_ylabel("Holiday", fontsize=11)
axes[2].set_xlabel("Date", fontsize=11)
axes[2].set_yticks([0, 1])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observation: publishing events (orange) should coincide with visitor spikes.")

## 3. Train/Test Split and Forecast Setup

We reserve the last 28 days for evaluation. The model will explain its 28-day forecast.

NeuralForecast requires:
- Training data: `unique_id`, `ds`, `y`, plus any exogenous columns
- Future exogenous data (`futr_df`): `unique_id`, `ds`, plus the exogenous columns (no `y`)

In [ ]:
HORIZON = 28
INPUT_SIZE = 56  # 2x horizon

train = df.iloc[:-HORIZON].copy()
test  = df.iloc[-HORIZON:].copy()

# futr_df: exogenous features for the forecast window (no y column)
futr_df = test[["unique_id", "ds", "published", "is_holiday"]].copy()

print(f"Training days:  {len(train)} ({train['ds'].min().date()} to {train['ds'].max().date()})")
print(f"Test days:      {len(test)} ({test['ds'].min().date()} to {test['ds'].max().date()})")
print(f"Horizon:        {HORIZON} days")
print(f"Input size:     {INPUT_SIZE} lags")
print()
print("futr_df sample:")
print(futr_df.head())

## 4. Train NHITS with Exogenous Features

Key parameters:
- `h=28`: forecast 28 days ahead
- `input_size=56`: use 56 past days as input
- `futr_exog_list=["published", "is_holiday"]`: tell NHITS which columns are known future covariates
- `max_steps=300`: enough steps for synthetic data (use 1000+ for real datasets)
- `val_size=28`: hold out the last 28 training days for validation

In [ ]:
models = [
    NHITS(
        h=INPUT_SIZE // 2,      # h = 28
        input_size=INPUT_SIZE,   # input_size = 56
        futr_exog_list=["published", "is_holiday"],
        max_steps=300,
        loss=MSE(),
        valid_loss=MAE()
    )
]

nf = NeuralForecast(models=models, freq="D")

print("Training NHITS...")
nf.fit(df=train, val_size=HORIZON)
print("Training complete.")

## 5. Call .explain() with Integrated Gradients

`.explain()` runs the model forward pass multiple times (20–300 steps for IG) to integrate gradients along the path from baseline to actual input. It returns both the forecast and the attribution dictionary.

The baseline is zero-filled by default, appropriate for binary features where 0 means "no event."

In [ ]:
print("Running .explain() with Integrated Gradients...")
fcsts_df, explanations = nf.explain(
    futr_df=futr_df,
    explainer="IntegratedGradients"
)

print("Done.")
print()
print("fcsts_df columns:", list(fcsts_df.columns))
print("explanations keys:", list(explanations.keys()))
print()

# Extract tensors
insample      = explanations["insample"]
futr_exog     = explanations["futr_exog"]
baseline_pred = explanations["baseline_predictions"]

print("Tensor shapes:")
print(f"  insample:             {insample.shape}")
print(f"  futr_exog:            {futr_exog.shape}")
print(f"  baseline_predictions: {baseline_pred.shape}")

## 6. Extract Insample Attributions

The `insample` tensor has shape `[batch, horizon, series, output, input_size, 2]`.

- Batch index: 0 (single forecast)
- Horizon: 28 forecast steps (0 to 27)
- Series: 0 (single blog series)
- Output: 0 (point forecast)
- Input size: 56 lag positions (0 = most recent, 55 = oldest)
- Last dim: [0] = lag value, [1] = attribution score

In [ ]:
# Attribution scores: shape (horizon=28, input_size=56)
# insample[batch, horizon, series, output, lag_position, attribution_index]
lag_attr = insample[0, :, 0, 0, :, 1]  # shape (28, 56)
lag_vals = insample[0, :, 0, 0, :, 0]  # shape (28, 56) — actual lag values

print("Lag attribution matrix shape:", lag_attr.shape)
print("  Rows: forecast step (0=day1, 27=day28)")
print("  Cols: lag position (0=most recent, 55=oldest)")
print()

# Most important lag for each forecast step
most_important_lag = np.argmax(np.abs(lag_attr), axis=1)
print("Most important lag per forecast step:")
for step in range(HORIZON):
    lag_idx = most_important_lag[step]
    attr_val = lag_attr[step, lag_idx]
    print(f"  Forecast day {step+1:2d}: lag {lag_idx:2d}  (attribution={attr_val:+.1f})")

## 7. Heatmap: Lag Attribution Across the Horizon

The heatmap rows are forecast steps (day 1 to day 28) and columns are lag positions (lag 0 = most recent observation). Red means the model uses that lag to push the forecast up; blue means it pushes the forecast down.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Symmetric color scale around zero
abs_max = np.abs(lag_attr).max()
im = ax.imshow(
    lag_attr,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-abs_max,
    vmax=abs_max,
    interpolation="nearest"
)

plt.colorbar(im, ax=ax, label="Attribution Score (red = raises forecast, blue = lowers)")

ax.set_xlabel("Lag Position (0 = most recent observation)", fontsize=12)
ax.set_ylabel("Forecast Step", fontsize=12)
ax.set_title(
    "NHITS Insample Attribution Heatmap\n"
    "Blog Traffic: which past lags drive each forecast step?",
    fontsize=13
)

# Mark every 7th lag to highlight weekly seasonality
tick_positions = [0, 7, 14, 21, 28, 35, 42, 49, 55]
ax.set_xticks(tick_positions)
ax.set_xticklabels([f"lag {x}" for x in tick_positions])

# Forecast step labels on y axis
ax.set_yticks(range(0, HORIZON, 7))
ax.set_yticklabels([f"day {i+1}" for i in range(0, HORIZON, 7)])

plt.tight_layout()
plt.show()

print("Reading the heatmap:")
print("- Dominant color at lag 0 = model weights recent observations heavily")
print("- Bands at lags 7, 14, 21 = weekly seasonality captured")
print("- Near-zero elsewhere = older lags not informative")

## 8. Summarize Attribution by Lag Quartile

Aggregate attributions across forecast steps and group lags into four windows to summarize the model's temporal attention.

In [ ]:
# Mean absolute attribution per lag, averaged across all 28 forecast steps
mean_abs_per_lag = np.abs(lag_attr).mean(axis=0)  # shape (56,)

# Group into windows
windows = {
    "Lag 0–6 (last week)": mean_abs_per_lag[0:7].mean(),
    "Lag 7–13 (week 2)": mean_abs_per_lag[7:14].mean(),
    "Lag 14–27 (weeks 3-4)": mean_abs_per_lag[14:28].mean(),
    "Lag 28–55 (weeks 5-8)": mean_abs_per_lag[28:56].mean(),
}

fig, ax = plt.subplots(figsize=(10, 5))
labels = list(windows.keys())
values = list(windows.values())
colors = ["#d32f2f", "#f57c00", "#388e3c", "#1565c0"]

bars = ax.bar(labels, values, color=colors, edgecolor="black", alpha=0.85)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005 * max(values),
        f"{val:.1f}",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.set_ylabel("Mean Absolute Attribution", fontsize=12)
ax.set_title("Attribution by Lag Window\nNHITS Blog Traffic — Integrated Gradients", fontsize=13)
ax.grid(True, alpha=0.3, axis="y")
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.show()

print("Expected: most attribution concentrated in Lag 0-6 (last week).")
print("Weekly seasonal patterns appear as elevated Lag 7-13.")

## 9. Check the Completeness Guarantee

For Integrated Gradients, the sum of all attributions should equal the difference between the actual forecast and the baseline prediction. A small gap (< 2%) confirms the attribution pipeline is working correctly.

In [ ]:
print("Completeness check — Integrated Gradients guarantee:")
print("sum(attributions) ≈ f(x) - f(x_baseline)")
print()

forecast_col = "NHITS"

for step in [0, 6, 13, 27]:  # spot-check 4 forecast steps
    predicted  = float(fcsts_df[forecast_col].iloc[step])
    baseline   = float(baseline_pred[0, step, 0, 0])
    
    insample_total = float(insample[0, step, 0, 0, :, 1].sum())
    exog_total     = float(futr_exog[0, step, 0, 0, :, :].sum())
    
    reconstructed = baseline + insample_total + exog_total
    gap_pct = abs(predicted - reconstructed) / (abs(predicted) + 1e-8) * 100
    
    print(f"Forecast day {step+1:2d}:")
    print(f"  Predicted:        {predicted:8.1f}")
    print(f"  Baseline:         {baseline:8.1f}")
    print(f"  Reconstructed:    {reconstructed:8.1f}")
    print(f"  Gap:              {abs(predicted - reconstructed):8.2f}  ({gap_pct:.2f}%)")
    print()

## Summary

In this notebook you:

1. Generated 1000-day synthetic blog traffic data with publishing events, holidays, weekly seasonality, and trend.
2. Trained NHITS with `futr_exog_list=["published", "is_holiday"]` — the only change needed to enable explainability.
3. Called `.explain(explainer="IntegratedGradients")` and parsed the returned `explanations` dictionary.
4. Extracted `insample` attributions (shape `[1, 28, 1, 1, 56, 2]`) and found that lag 0 (most recent observation) consistently receives the highest attribution.
5. Built a heatmap showing how the model's attention to past lags changes across the 28-day horizon.
6. Verified the IG completeness guarantee: reconstructed forecast ≈ actual forecast.

**Next:** `02_attribution_visualization.ipynb` — compare all three methods, waterfall plots, business narrative.